# Qui va partir ? Prédiction du churn Revolut
**Binôme :** [Nom 1] — [Nom 2]  
**Track A** — ML for Business (MSA-DATB07-08)

**Recommandation (1 phrase) :** _à écrire à la fin_

In [ ]:
# 0. Vérification des fichiers : le notebook doit tourner depuis ce dossier
from pathlib import Path

FILES = ['rev-users.csv', 'rev-transactions.csv', 'rev-notifications.csv']
missing = [f for f in FILES if not Path(f).exists()]
assert not missing, f'Fichiers manquants : {missing}'
print('OK : tous les fichiers sont présents')

## 1. Cadrage (avant tout modèle)

| Question | Réponse |
|---|---|
| Qui décide | l'équipe rétention |
| Unité | un client **actif** à la date de coupure (au moins une transaction dans les 60 jours avant) |
| Moment de la décision | à la date de coupure, une fois par mois |
| Ce qu'on fait de la sortie | on classe les clients par risque de départ `p` et on envoie l'offre aux premiers de la liste |
| Contrainte | budget de `BUDGET` offres par mois (hypothèse) |
| Fausse alerte | on paie l'offre `X` pour un client qui serait resté |
| Client raté | on perd sa valeur `Y` |

**Règle de décision.** Envoyer l'offre à un client rapporte en moyenne `p × Z × Y − X` :
chance qu'il parte × chance que l'offre le retienne × ce qu'il rapporte − prix de l'offre.
On envoie si ce gain est positif, c'est-à-dire si **p > X / (Z × Y)**.

| Chiffre | Valeur | Origine |
|---|---|---|
| `Y` valeur d'un client retenu (12 mois) | calculée ci-dessous | **données** (frais + commission carte) |
| `X` coût de l'offre | 5 € | **hypothèse** : un mois de Premium offert, coût réel pour Revolut inférieur au prix affiché |
| `Z` taux de réussite de l'offre | inconnu | **absent des données** (aucune offre envoyée) → on calcule le Z de point mort |
| `BUDGET` | 500 offres/mois | **hypothèse** à confirmer avec l'équipe |

## 2. Définition du churn

**Règle :** un client actif (au moins une transaction dans les 60 jours avant la coupure) est un churner s'il n'a **aucune transaction dans les 60 jours après**.

**Pourquoi 60 jours :** plus la fenêtre est courte, plus on étiquette comme partis des clients simplement en pause (cellule de vérification plus bas) :
- à 45 jours, 31 % des churners reviennent dans le mois suivant ;
- à 60 jours, 26 % ;
- 90 jours serait plus propre, mais repousserait toutes les coupures avant l'inscription d'une partie des clients.

→ Nous gardons 60 jours ; le taux de faux churn est une **limite assumée**.

**Fin des données :** les transactions s'arrêtent le 16 mai 2019, donc la fenêtre du label ne peut pas dépasser cette date.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

users = pd.read_csv('rev-users.csv', parse_dates=['created_date'])
notifs = pd.read_csv('rev-notifications.csv', parse_dates=['created_date'])
tx = pd.read_csv('rev-transactions.csv', parse_dates=['created_date'])
print(users.shape, notifs.shape, tx.shape)
print('Transactions :', tx.created_date.min(), '->', tx.created_date.max())

### 2.1 Vérification de la fenêtre : combien de « churners » reviennent ?

In [ ]:
cutoff = pd.Timestamp('2019-01-15')

for w in [30, 45, 60, 90]:
    W = pd.Timedelta(days=w)
    before = tx[(tx.created_date >= cutoff - W) & (tx.created_date < cutoff)]
    after = tx[(tx.created_date >= cutoff) & (tx.created_date < cutoff + W)]
    actifs = set(before.user_id)
    churners = actifs - set(after.user_id)
    # reviennent dans les 30 jours qui suivent la fin de la fenêtre
    retour = tx[(tx.created_date >= cutoff + W) & (tx.created_date < cutoff + W + pd.Timedelta(days=30))]
    reviennent = churners & set(retour.user_id)
    print(f'{w} j : churn {len(churners)/len(actifs):.1%}, '
          f'reviennent dans le mois {len(reviennent)/len(churners):.1%}')

### 2.2 Dates de coupure (découpage dans le temps)

| Partie | Coupure | Fenêtre du label |
|---|---|---|
| Fit | 15/01/2019 | → 16/03/2019 |
| Validation | 15/02/2019 | → 16/04/2019 |
| Test (ouvert une seule fois) | 15/03/2019 | → 14/05/2019 |

In [ ]:
WINDOW = pd.Timedelta(days=60)
CUTOFFS = {
    'fit': pd.Timestamp('2019-01-15'),
    'validation': pd.Timestamp('2019-02-15'),
    'test': pd.Timestamp('2019-03-15'),
}
last_tx = tx.created_date.max()

for name, c in CUTOFFS.items():
    assert c + WINDOW <= last_tx, f'{name} : la fenêtre du label dépasse la fin des données'
    print(f'{name:<11} coupure {c.date()}  label jusqu\'au {(c + WINDOW).date()}')

# Nouveaux clients : ancienneté minimale à chaque coupure
last_signup = users.created_date.max()
for name, c in CUTOFFS.items():
    print(f'{name:<11} dernier inscrit il y a {(c - last_signup).days} jours')

### 2.3 Valeur d'un client retenu `Y` (calculée sur les données)

Revenu observable par client actif et par mois, sur janvier–avril 2019, transactions terminées :
- les frais facturés (`FEE`) ;
- une commission sur les paiements carte : **0,3 %** (hypothèse, l'interchange européen est plafonné à 0,2 % en débit et 0,3 % en crédit).

Ce revenu sous-estime la valeur réelle (abonnements, change, etc. non visibles) : `Y` est donc **prudent**.

In [ ]:
INTERCHANGE = 0.003   # hypothèse : commission sur paiement carte
USD_TO_EUR = 0.89     # hypothèse : taux moyen début 2019

periode = tx[(tx.created_date >= '2019-01-01') & (tx.created_date < '2019-05-01')
             & (tx.transactions_state == 'COMPLETED')]
n_clients = periode.user_id.nunique()
n_mois = 4

frais = periode.loc[periode.transactions_type == 'FEE', 'amount_usd'].sum()
carte = periode.loc[periode.transactions_type == 'CARD_PAYMENT', 'amount_usd'].sum()

revenu_mois_usd = (frais + INTERCHANGE * carte) / n_clients / n_mois
Y = revenu_mois_usd * 12 * USD_TO_EUR
print(f'Revenu par client actif : {revenu_mois_usd:.2f} $/mois  ->  Y = {Y:.1f} € sur 12 mois')

### 2.4 Coût de l'offre, point mort et sensibilité

`Z` n'est pas dans les données. Au lieu de le deviner, on calcule le **Z minimum** pour que l'offre soit rentable, même sur un client sûr de partir (`p = 1`) : `Z > X / Y`.

In [ ]:
X = 5          # hypothèse : coût d'une offre en €
BUDGET = 500   # hypothèse : offres par mois

Z_point_mort = X / Y
print(f'Z de point mort : {Z_point_mort:.0%}  (en dessous, aucun modèle ne rend l\'offre rentable)')

# Sensibilité : seuil de risque p au-dessus duquel on envoie l'offre
rows = []
for x in [2, 5, 8]:
    for z in [0.25, 0.35, 0.50]:
        seuil = x / (z * Y)
        rows.append({'X (€)': x, 'Z': f'{z:.0%}',
                     'seuil p': round(seuil, 2) if seuil <= 1 else 'jamais rentable'})
pd.DataFrame(rows).pivot(index='X (€)', columns='Z', values='seuil p')

## 3. Audit des données
_Lignes impossibles, colonnes constantes, valeurs manquantes, clients sans transaction._

## 4. Découpage dans le temps : fit / validation / test
| Partie | Coupure | Fenêtre du label |
|---|---|---|
| Fit | [date] | [date] → [date] |
| Validation | [date] | [date] → [date] |
| Test (ouvert une fois) | [date] | [date] → [date] |

## 5. Features + test anti-fuite
| Colonne | Connue au moment de décider ? | Décision |
|---|---|---|
| `plan` | non : photo prise à l'export | retirée |
| `num_referrals`, `num_successful_referrals` | toujours 0 | retirées |
| ... | ... | ... |

## 6. Baseline vs modèles
- Baseline : _règle en une ligne_
- Régression logistique
- Modèle à base d'arbres

Métriques : PR-AUC (comparée au taux de churn) + € pour 1000 clients. Pas d'accuracy.

## 7. Seuil fixé par les coûts et budget (choisi sur la validation)

## 8. Test final — **ouvert une seule fois ici**

## 9. Où le modèle se trompe
_Tableau ou graphe : pires erreurs, groupes faibles (pays, ancienneté, activité)._

## 10. Limites et recommandation
- Ce que les données ne contiennent pas : _..._
- Hypothèses : _..._
- **Décision :** on lance / on ne lance pas / on lance si _..._
- À surveiller le premier mois : _..._